# ResNet Trigger Node: Worker Round + Manager Cycle

This notebook now matches the current node design more closely.

It has two layers:

1. A **single worker-node training round** using the same `prepare.py`, `resnet_1d.py`, and `train.py` split as the node.
2. An optional **manager/control-plane cycle** that goes through the harness API and exercises `/baseline`, `/run`, `/keep` or `/discard`.

What is included here:

- direct worker execution on this node
- AUC-based model selection
- best-model artifact saving
- optional manager-level orchestration through the harness scripts

What is not this notebook's main job:

- it is not the production manager itself
- it is not the final multi-node claw orchestration notebook

The actual worker execution surface remains `train.py`.


In [1]:
import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

# Prefer Apple Metal for speed on this notebook run.
os.environ.setdefault("RESNET_TRIGGER_DEVICE", "mps")

from prepare import ARTIFACT_DIR, DataConfig, prepare_run_arrays, resolve_device

SEED = 123
torch.manual_seed(SEED)

DATA_DIR = Path.cwd()
NODE_ROOT = DATA_DIR
REPO_ROOT = NODE_ROOT.parents[1]
CLAW_ROOT = REPO_ROOT / "harness" / "claw-code"
CYCLE_SCRIPT = CLAW_ROOT / "scripts" / "run_resnet_trigger_cycle.py"
DEMO_MANAGER_SCRIPT = CLAW_ROOT / "scripts" / "run_demo_manager.py"
CLAW_BINARY = CLAW_ROOT / "target" / "debug" / "claw"

CFG = {
    "signal_file": "signal_vacuum_sum_crop_4000x8000.h5",
    "noise_file": "noise_traces_4000x8000.h5",
    "n_signal": 4000,
    "n_noise": 4000,
    "trace_len": 8000,
    "train_frac": 0.70,
    "val_frac": 0.15,
    "test_frac": 0.15,
    "batch_size": 32,
    "epochs": 5,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "eps": 1e-6,
    "layers": [1, 1, 1],
    "kernel_size": 7,
    "grad_clip_norm": 1.0,
}

DATA_CFG = DataConfig(
    signal_file=CFG["signal_file"],
    noise_file=CFG["noise_file"],
    n_signal=CFG["n_signal"],
    n_noise=CFG["n_noise"],
    trace_len=CFG["trace_len"],
    train_frac=CFG["train_frac"],
    val_frac=CFG["val_frac"],
    eps=CFG["eps"],
)

PREFERRED_DEVICE = resolve_device()
WORKER_DEVICE = torch.device("cpu") if PREFERRED_DEVICE.type == "mps" else PREFERRED_DEVICE

print("preferred_device:", PREFERRED_DEVICE)
print("worker_device:", WORKER_DEVICE)
print(json.dumps(CFG, indent=2))
print("node_root:", NODE_ROOT)
print("artifacts_dir:", ARTIFACT_DIR)
print("claw_root:", CLAW_ROOT)


preferred_device: mps
worker_device: cpu
{
  "signal_file": "signal_vacuum_sum_crop_4000x8000.h5",
  "noise_file": "noise_traces_4000x8000.h5",
  "n_signal": 4000,
  "n_noise": 4000,
  "trace_len": 8000,
  "train_frac": 0.7,
  "val_frac": 0.15,
  "test_frac": 0.15,
  "batch_size": 32,
  "epochs": 5,
  "learning_rate": 0.0005,
  "weight_decay": 0.0001,
  "eps": 1e-06,
  "layers": [
    1,
    1,
    1
  ],
  "kernel_size": 7,
  "grad_clip_norm": 1.0
}
node_root: /Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger
artifacts_dir: /Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger/artifacts
claw_root: /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code


In [2]:
# Inspect the node H5 sources through the shared helper layer.
from prepare import load_source_arrays

signal_all, noise_all, source_summary = load_source_arrays(DATA_CFG)
signal_path = DATA_DIR / CFG["signal_file"]
noise_path = DATA_DIR / CFG["noise_file"]

print("Signal file:", signal_path)
print("Noise file:", noise_path)
print(json.dumps(source_summary, indent=2))
print("Aligned shapes -> signal:", signal_all.shape, "noise:", noise_all.shape)


Signal file: /Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger/signal_vacuum_sum_crop_4000x8000.h5
Noise file: /Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger/noise_traces_4000x8000.h5
{
  "signal_file": "/Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger/signal_vacuum_sum_crop_4000x8000.h5",
  "noise_file": "/Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger/noise_traces_4000x8000.h5",
  "signal_key": "traces",
  "noise_key": "traces",
  "signal_shape": [
    4000,
    8000
  ],
  "noise_shape": [
    4000,
    8000
  ],
  "trace_len": 8000
}
Aligned shapes -> signal: (4000, 8000) noise: (4000, 8000)


### Worker Round

The next cells run one worker-node round only.

That means:

- one dataset split using one seed
- one model configuration
- five training epochs
- best epoch chosen by validation AUC
- best checkpoint and metrics saved to the node artifacts directory

This is much closer to how the actual worker behaves than the earlier 3-seed notebook batch.


In [3]:
# Reuse the node split helper and keep a small notebook wrapper for convenience.
def prepare_run_datasets(run_seed):
    prep_t0 = time.perf_counter()
    X_train, y_train, X_val, y_val, X_test, y_test, split_meta = prepare_run_arrays(DATA_CFG, run_seed=run_seed)
    dataset_prep_seconds = time.perf_counter() - prep_t0
    return X_train, y_train, X_val, y_val, X_test, y_test, dataset_prep_seconds, split_meta


In [4]:
# Import the shared node backbone directly.
from resnet_1d import ResNet1D
import torch.nn as nn


def build_model():
    norm_layer = nn.BatchNorm1d if WORKER_DEVICE.type == "mps" else None
    model = ResNet1D(
        in_channels=1,
        layers=CFG["layers"],
        classes=1,
        kernel_size=CFG["kernel_size"],
        norm_layer=norm_layer,
    )
    return model.to(WORKER_DEVICE)


tmp_model = build_model()
param_count = int(sum(p.numel() for p in tmp_model.parameters()))
del tmp_model
print("parameter_count:", param_count)


parameter_count: 961857


In [5]:
# Single worker-node round with AUC-based selection.
def eval_loader(model, loader, criterion):
    model.eval()
    losses = []
    ys = []
    probs = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(WORKER_DEVICE, dtype=torch.float32, non_blocking=True)
            yb = yb.to(WORKER_DEVICE, dtype=torch.float32, non_blocking=True).view(-1, 1)
            logits = model(xb)
            if not torch.isfinite(logits).all():
                raise RuntimeError("Non-finite logits detected during evaluation")
            logits_for_loss = logits.float().clamp(-30.0, 30.0)
            loss = criterion(logits_for_loss, yb)
            losses.append(float(loss.item()))
            ys.append(yb.detach().cpu().numpy().ravel())
            probs.append(torch.sigmoid(logits_for_loss).detach().cpu().numpy().ravel())
    y_true = np.concatenate(ys)
    y_prob = np.concatenate(probs)
    return float(np.mean(losses)), y_true, y_prob

run_seed = SEED
torch.manual_seed(run_seed)

(X_train, y_train, X_val, y_val, X_test, y_test, dataset_prep_seconds, split_meta) = prepare_run_datasets(run_seed)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=0)

model = build_model()
criterion = torch.nn.BCEWithLogitsLoss()
optimizer_learning_rate = min(CFG["learning_rate"], 5e-4) if WORKER_DEVICE.type == "mps" else CFG["learning_rate"]
optimizer_eps = 1e-4 if WORKER_DEVICE.type == "mps" else 1e-8
optimizer = torch.optim.AdamW(model.parameters(), lr=optimizer_learning_rate, weight_decay=CFG["weight_decay"], eps=optimizer_eps)

history = []
total_t0 = time.perf_counter()
train_time_sum = 0.0
val_time_sum = 0.0
best_epoch = None
best_model_path = ARTIFACT_DIR / "notebook_best_model.pt"
best_metrics_path = ARTIFACT_DIR / "notebook_best_performance.json"

print(f"Worker round seed={run_seed}")
print(f"train/val/test: {X_train.shape[0]}/{X_val.shape[0]}/{X_test.shape[0]}")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    t0 = time.perf_counter()
    train_losses = []
    for xb, yb in train_loader:
        xb = xb.to(WORKER_DEVICE, dtype=torch.float32, non_blocking=True)
        yb = yb.to(WORKER_DEVICE, dtype=torch.float32, non_blocking=True).view(-1, 1)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        if not torch.isfinite(logits).all():
            raise RuntimeError("Non-finite logits detected during training")
        logits_for_loss = logits.float().clamp(-30.0, 30.0)
        loss = criterion(logits_for_loss, yb)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss detected during training")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
        optimizer.step()
        train_losses.append(float(loss.item()))
    train_time_sum += time.perf_counter() - t0

    v0 = time.perf_counter()
    val_loss, y_val_true, y_val_prob = eval_loader(model, val_loader, criterion)
    val_time_sum += time.perf_counter() - v0

    row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "val_loss": val_loss,
        "val_auc": float(roc_auc_score(y_val_true, y_val_prob)),
        "val_roc_auc": float(roc_auc_score(y_val_true, y_val_prob)),
        "val_pr_auc": float(average_precision_score(y_val_true, y_val_prob)),
    }
    history.append(row)
    print(row)

    if best_epoch is None or (row["val_auc"], row["val_pr_auc"], -row["val_loss"]) > (best_epoch["val_auc"], best_epoch["val_pr_auc"], -best_epoch["val_loss"]):
        best_epoch = row
        torch.save(model.state_dict(), best_model_path)
        best_metrics_path.write_text(json.dumps(best_epoch, indent=2))

assert best_epoch is not None

test_loss, y_test_true, y_test_prob = eval_loader(model, test_loader, criterion)
y_test_pred = (y_test_prob >= 0.5).astype(np.int64)
test_cm = confusion_matrix(y_test_true.astype(np.int64), y_test_pred).tolist()
total_wall_seconds = time.perf_counter() - total_t0

worker_summary = {
    "best_epoch": int(best_epoch["epoch"]),
    "best_val_auc": float(best_epoch["val_auc"]),
    "best_val_roc_auc": float(best_epoch["val_roc_auc"]),
    "best_val_pr_auc": float(best_epoch["val_pr_auc"]),
    "best_val_loss": float(best_epoch["val_loss"]),
    "test_loss": float(test_loss),
    "test_roc_auc": float(roc_auc_score(y_test_true, y_test_prob)),
    "test_pr_auc": float(average_precision_score(y_test_true, y_test_prob)),
    "test_confusion_matrix_threshold_0_5": test_cm,
    "parameter_count": int(sum(p.numel() for p in model.parameters())),
    "device": str(WORKER_DEVICE),
    "dataset_prep_seconds": float(dataset_prep_seconds),
    "training_seconds": float(train_time_sum),
    "validation_seconds": float(val_time_sum),
    "total_wall_seconds": float(total_wall_seconds),
    "best_model_path": str(best_model_path),
    "best_metrics_path": str(best_metrics_path),
}

history_path = ARTIFACT_DIR / "notebook_history_single_run.json"
summary_path = ARTIFACT_DIR / "notebook_worker_summary.json"
history_path.write_text(json.dumps(history, indent=2))
summary_path.write_text(json.dumps(worker_summary, indent=2))

print("\nWorker round summary:")
print(json.dumps(worker_summary, indent=2))


Worker round seed=123
train/val/test: 5600/1200/1200
{'epoch': 1, 'train_loss': 0.5964412374155862, 'val_loss': 0.4719503349379489, 'val_auc': 0.8729388888888889, 'val_roc_auc': 0.8729388888888889, 'val_pr_auc': 0.8969877060483515}
{'epoch': 2, 'train_loss': 0.5192041388579778, 'val_loss': 0.4377395047953254, 'val_auc': 0.878886111111111, 'val_roc_auc': 0.878886111111111, 'val_pr_auc': 0.9024663212949554}
{'epoch': 3, 'train_loss': 0.47197583181517466, 'val_loss': 0.4673002456363879, 'val_auc': 0.8823222222222222, 'val_roc_auc': 0.8823222222222222, 'val_pr_auc': 0.9055134057578448}
{'epoch': 4, 'train_loss': 0.45211021082741876, 'val_loss': 0.4099830741945066, 'val_auc': 0.8871944444444445, 'val_roc_auc': 0.8871944444444445, 'val_pr_auc': 0.9099859324179869}
{'epoch': 5, 'train_loss': 0.45093214631080625, 'val_loss': 0.4059231359707682, 'val_auc': 0.8885861111111111, 'val_roc_auc': 0.8885861111111111, 'val_pr_auc': 0.9109412527365148}

Worker round summary:
{
  "best_epoch": 5,
  "best

### Manager / Control Plane

The next section introduces the manager layer that the earlier notebook did not have.

This is the stack used here:

- worker node: this `ResNet_trigger` directory
- control plane: `harness/claw-code/src/api_server.py`
- manager stub: `harness/claw-code/scripts/run_resnet_trigger_cycle.py`

That manager-side script performs:

- API server startup
- `/baseline`
- `/run`
- `/keep` or `/discard`
- status and memory inspection

What you need from your machine if you want to run it live:

- `uv` installed
- the node environment synced
- your worker backend running, for example Ollama at `http://localhost:11434`
- a local model name, for example `qwen2.5-coder:7b`


In [ ]:
# Manager-cycle configuration.
RUN_MANAGER_CYCLE = True
BACKEND = "ollama"
MODEL = "qwen2.5-coder:7b"
HOST = "http://localhost:11434"
TIMEOUT_SECONDS = 1800
CANDIDATE_LEARNING_RATE = 5e-4

manager_command = [
    sys.executable,
    str(CYCLE_SCRIPT),
    "--node-root", str(NODE_ROOT),
    "--backend", BACKEND,
    "--model", MODEL,
    "--host", HOST,
    "--device", str(WORKER_DEVICE),
    "--timeout-seconds", str(TIMEOUT_SECONDS),
    "--candidate-learning-rate", str(CANDIDATE_LEARNING_RATE),
]

print("Manager cycle command:")
print(shlex.join(manager_command))
print("run_manager_cycle:", RUN_MANAGER_CYCLE)


Manager cycle command:
/Users/wongdowling/opt/anaconda3/bin/python /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/scripts/run_resnet_trigger_cycle.py --node-root /Users/wongdowling/Documents/autoresearch_harness/nodes/ResNet_trigger --backend ollama --model qwen2.5-coder:7b --host http://localhost:11434 --device cpu --timeout-seconds 1800 --candidate-learning-rate 0.0005
run_manager_cycle: False


In [7]:
# Optional real manager/control-plane cycle.
# Set RUN_MANAGER_CYCLE = True in the previous cell once your local model backend is running.
manager_cycle_summary = None

if RUN_MANAGER_CYCLE:
    result = subprocess.run(
        manager_command,
        cwd=CLAW_ROOT,
        text=True,
        capture_output=True,
    )
    print("returncode:", result.returncode)
    if result.stdout.strip():
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Manager cycle failed")
    manager_cycle_summary = json.loads(result.stdout)
    print("\nParsed manager cycle summary keys:")
    print(sorted(manager_cycle_summary.keys()))
else:
    print("Manager cycle not executed. Start your backend, then set RUN_MANAGER_CYCLE = True.")


Manager cycle not executed. Start your backend, then set RUN_MANAGER_CYCLE = True.


### Claw Handoff

The notebook now includes a real manager/control-plane path through the harness scripts.

If you want to move one step closer to the actual claw-manager level, the next command to care about is the harness-side manager invocation. This is kept as an explicit command preview because it depends on your built `claw` binary and your chosen manager model.


In [8]:
claw_command = [
    "/bin/zsh",
    "-lc",
    (
        "OPENAI_BASE_URL=http://localhost:11434/v1 "
        "OPENAI_API_KEY=local "
        "ANTHROPIC_API_KEY= "
        "ANTHROPIC_AUTH_TOKEN= "
        f"{CLAW_BINARY} --output-format json --model {MODEL} prompt "
        '"Use the autoresearch API on this machine as manager. First inspect /status and /memory, then propose one bounded experiment."'
    ),
]

print("Optional claw command preview:")
print(shlex.join(claw_command))
print("claw_binary_exists:", CLAW_BINARY.exists())


Optional claw command preview:
/bin/zsh -lc 'OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=local ANTHROPIC_API_KEY= ANTHROPIC_AUTH_TOKEN= /Users/wongdowling/Documents/autoresearch_harness/harness/claw-code/target/debug/claw --output-format json --model qwen2.5-coder:7b prompt "Use the autoresearch API on this machine as manager. First inspect /status and /memory, then propose one bounded experiment."'
claw_binary_exists: False
